In [ ]:
# IMPORTS

from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import wfdb

import tensorflow as tf

from keras.models import Sequential
from keras.layers import LSTM, GRU, Dense, Dropout, BatchNormalization
from keras.callbacks import EarlyStopping, ModelCheckpoint

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix)

print("All imports successful!")

In [ ]:
# CONFIGURATION

# Data parameters
NUM_PATIENTS = 20
RECORDINGS_PER_PATIENT = 30
MIN_RECORDINGS = 40

# Signal parameters
FS = 500
WINDOW_SIZE = 2000
STEP_SIZE = 750

# Model parameters 
LSTM_UNITS_1 = 128
LSTM_UNITS_2 = 64
GRU_UNITS_1 = 128
GRU_UNITS_2 = 64
DENSE_UNITS = 64
DROPOUT_RATE = 0.3
DENSE_DROPOUT = 0.2

# Training parameters 
EPOCHS = 100
BATCH_SIZE = 32
VALIDATION_SPLIT = 0.2
PATIENCE = 15
TEST_SIZE = 0.2

# NaN handling 
NAN_THRESHOLD = 0.05
MAX_RECHECK_ITERATIONS = 10

# Paths 
BASE_PATH = 'C:/Users/Robert/Marisiensis2026/mimic-iv-dataset/files'
CSV_PATH = 'C:/Users/Robert/Marisiensis2026/mimic-iv-dataset/record_list.csv'



In [ ]:
# LOADING AND FILTERING THE CSV

# Load the CSV 
df = pd.read_csv(CSV_PATH)
print(f"Total recordings in dataset: {len(df)}")
print(f"Total unique patients: {df['subject_id'].nunique()}")

# Find patients with enough recordings 
recording_counts = df.groupby('subject_id').size()
eligible_patients = recording_counts[recording_counts >= MIN_RECORDINGS].index
print(f"\nPatients with at least {MIN_RECORDINGS} recordings: {len(eligible_patients)}")

# Select top NUM_PATIENTS by recording count 
top_patients = (recording_counts[eligible_patients]
                .sort_values(ascending=False)
                .head(NUM_PATIENTS)
                .index)

print(f"\nSelected {NUM_PATIENTS} patients:")
for patient in top_patients:
    print(f"  Patient {patient}: {recording_counts[patient]} recordings available")

# Filter dataframe to selected patients 
df_filtered = df[df['subject_id'].isin(top_patients)].copy()

# Cap at RECORDINGS_PER_PATIENT per patient 
df_filtered = (df_filtered
               .groupby('subject_id')
               .head(RECORDINGS_PER_PATIENT)
               .reset_index(drop=True))

print(f"\nFiltered dataset: {len(df_filtered)} recordings")
print("Recordings per patient:")
print(df_filtered.groupby('subject_id').size())

In [ ]:
# LOADING THE SIGNALS

all_signals = []
all_labels = []

# Track loaded paths to avoid duplicates during rebalancing later
already_loaded_paths = set(df_filtered['path'].tolist())

print(f"Loading {len(df_filtered)} recordings...")

for _, row in df_filtered.iterrows():
    record_path = f"{BASE_PATH}/{row['path'].replace('files/', '')}"
    
    try:
        record = wfdb.rdrecord(record_path)
        all_signals.append(record.p_signal)
        all_labels.append(row['subject_id'])
        
    except Exception as e:
        print(f"  Failed to load {record_path}: {e}")

print(f"\nSuccessfully loaded: {len(all_signals)} recordings")
print(f"Each signal shape: {all_signals[0].shape}")

# Initial balance check 
print("\nInitial recordings per patient:")
label_counts = Counter(all_labels)
for patient, count in sorted(label_counts.items()):
    shortfall = RECORDINGS_PER_PATIENT - count
    print(f"  Patient {patient}: {count} recordings "
          f"{'✓' if shortfall == 0 else f'— needs {shortfall} more'}")

In [ ]:
# NaN CHECKING, FIXING, DROPPING AND REBALANCING

import copy

# Copy only the loaded recordings, leaving originals intact 
all_signals_clean = copy.deepcopy(all_signals)
all_labels_clean = all_labels.copy()

print(f"Created working copies of {len(all_signals_clean)} recordings")
print("Original signals preserved intact in all_signals / all_labels")

iteration = 1

while True:
    print(f"\n{'='*40}")
    print(f"NaN Check — Iteration {iteration}")
    print(f"{'='*40}")

    recordings_to_drop = []
    recordings_to_fix = []

    for i, signal in enumerate(all_signals_clean):
        if np.isnan(signal).any():

            worst_lead_nan_pct = 0
            total_nans = 0
            for lead in range(signal.shape[1]):
                lead_nan_count = np.isnan(signal[:, lead]).sum()
                lead_nan_pct = lead_nan_count / signal.shape[0]
                worst_lead_nan_pct = max(worst_lead_nan_pct, lead_nan_pct)
                total_nans += lead_nan_count

            if worst_lead_nan_pct > NAN_THRESHOLD:
                recordings_to_drop.append(i)
                print(f"  Recording {i} (Patient {all_labels_clean[i]}): "
     f"{total_nans} NaN values, worst lead at "
     f"{worst_lead_nan_pct*100:.1f}% — will DROP")
            else:
                recordings_to_fix.append(i)
                print(f"  Recording {i} (Patient {all_labels_clean[i]}): "
     f"{total_nans} NaN values, worst lead at "
     f"{worst_lead_nan_pct*100:.1f}% — will FIX")

    if not recordings_to_drop and not recordings_to_fix:
        print("No NaN values found — working copy is clean!")
        break

    for i in recordings_to_fix:
        signal = all_signals_clean[i]
        for lead in range(signal.shape[1]):
            lead_mean = np.nanmean(signal[:, lead])
            nan_mask = np.isnan(signal[:, lead])
            signal[nan_mask, lead] = lead_mean
        all_signals_clean[i] = signal
        print(f"  Fixed recording {i} (Patient {all_labels_clean[i]}) in working copy")

    for i in sorted(recordings_to_drop, reverse=True):
        print(f"  Dropping recording {i} (Patient {all_labels_clean[i]}) from working copy")
        all_signals_clean.pop(i)
        all_labels_clean.pop(i)

    if recordings_to_drop:
        print(f"\nRebalancing after dropping {len(recordings_to_drop)} recordings...")

        label_counts = Counter(all_labels)
        target_count = RECORDINGS_PER_PATIENT

        patients_needing_more = {patient: target_count - count
                                 for patient, count in label_counts.items()
                                 if count < target_count}

        print(f"Current counts after dropping:")
        for patient, count in sorted(label_counts.items()):
            shortfall = target_count - count
            print(f"  Patient {patient}: {count} recordings "
                  f"{'✓' if shortfall == 0 else f'— needs {shortfall} more'}")

        for patient_id, shortfall in patients_needing_more.items():
            print(f"\n  Loading {shortfall} replacement(s) for Patient {patient_id}...")

            patient_df = df[df['subject_id'] == patient_id]
            unused = patient_df[~patient_df['path'].isin(already_loaded_paths)]

            if len(unused) == 0:
                print(f"  WARNING — No unused recordings left for Patient {patient_id}")
                continue

            loaded_count = 0
            for _, row in unused.iterrows():
                if loaded_count >= shortfall:
                    break

                record_path = f"{BASE_PATH}/{row['path'].replace('files/', '')}"
                try:
                    record = wfdb.rdrecord(record_path)
                    all_signals.append(record.p_signal)
                    all_labels.append(row['subject_id'])
                    already_loaded_paths.add(row['path'])
                    loaded_count += 1
                    print(f"    Loaded replacement "
                          f"{loaded_count}/{shortfall} for Patient {patient_id}")

                except Exception as e:
                    print(f"    Failed to load replacement: {e}")

    iteration += 1

    if iteration > MAX_RECHECK_ITERATIONS:
        print("WARNING — reached maximum iterations, stopping")
        break

# Final summary 
print(f"\n{'='*40}")
print(f"Final dataset summary")
print(f"{'='*40}")
print(f"Original recordings preserved: {len(all_signals)}")
print(f"Working copy recordings: {len(all_signals_clean)}")
print(f"Signal shape: {all_signals_clean[0].shape}")
print(f"\nFinal recordings per patient:")
for patient, count in sorted(Counter(all_labels_clean).items()):
    print(f"  Patient {patient}: {count} recordings "
          f"{'✓' if count == RECORDINGS_PER_PATIENT else '⚠ SHORT'}")

In [ ]:
# NORMALIZATION AND SEGMENTATION

# Normalize each recording and each lead independently
print("Normalizing signals...")

X_normalized = np.zeros_like(np.array(all_signals_clean))

for i in range(len(all_signals_clean)):
    for lead in range(all_signals_clean[0].shape[1]):
        lead_signal = all_signals_clean[i][:, lead]
        mean = np.mean(lead_signal)
        std = np.std(lead_signal)

        if std == 0:
            X_normalized[i, :, lead] = 0
        else:
            X_normalized[i, :, lead] = (lead_signal - mean) / std

print(f"Normalization complete!")
print(f"Sample mean: {np.mean(X_normalized[0, :, 0]):.6f} (should be ~0)")
print(f"Sample std:  {np.std(X_normalized[0, :, 0]):.6f} (should be ~1)")

# Segment into windows 
print(f"\nSegmenting into windows of {WINDOW_SIZE} samples "
      f"({WINDOW_SIZE/FS:.1f}s) with step {STEP_SIZE} samples "
      f"({STEP_SIZE/FS:.1f}s)...")

X_windows = []
y_windows = []

for i in range(len(X_normalized)):
    signal = X_normalized[i]
    label = all_labels_clean[i]

    start = 0
    while start + WINDOW_SIZE <= signal.shape[0]:
        X_windows.append(signal[start:start + WINDOW_SIZE])
        y_windows.append(label)
        start += STEP_SIZE

X_windows = np.array(X_windows)
y_windows = np.array(y_windows)

print(f"Segmentation complete!")
print(f"Total windows: {len(X_windows)}")
print(f"Windows shape: {X_windows.shape}")
print(f"\nWindows per patient:")
for patient in np.unique(y_windows):
    count = np.sum(y_windows == patient)
    print(f"  Patient {patient}: {count} windows")

# Encode labels 
print("\nEncoding labels...")

le = LabelEncoder()
y_encoded = le.fit_transform(y_windows)

print(f"Classes: {le.classes_}")
print(f"Encoded as: {list(range(len(le.classes_)))}")

# Train/test split 
print("\nSplitting into train and test sets...")

X_train, X_test, y_train, y_test = train_test_split(
    X_windows, y_encoded,
    test_size=TEST_SIZE,
    random_state=42,
    stratify=y_encoded
)

print(f"Training set:   {X_train.shape} — {len(y_train)} windows")
print(f"Test set:       {X_test.shape} — {len(y_test)} windows")
print(f"\nTraining windows per patient:")
for i in range(NUM_PATIENTS):
    print(f"  Patient {i}: {np.sum(y_train == i)} windows")
print(f"\nTest windows per patient:")
for i in range(NUM_PATIENTS):
    print(f"  Patient {i}: {np.sum(y_test == i)} windows")

In [ ]:
# BUILDING THE MODELS

input_shape = (X_train.shape[1], X_train.shape[2])
num_classes = len(np.unique(y_encoded))

print(f"Input shape: {input_shape}")
print(f"Number of classes: {num_classes}")

# LSTM Model 
def build_lstm_model(input_shape, num_classes):
    model = Sequential([
        LSTM(LSTM_UNITS_1, return_sequences=True, input_shape=input_shape),
        BatchNormalization(),
        Dropout(DROPOUT_RATE),

        LSTM(LSTM_UNITS_2, return_sequences=False),
        BatchNormalization(),
        Dropout(DROPOUT_RATE),

        Dense(DENSE_UNITS, activation='relu'),
        Dropout(DENSE_DROPOUT),

        Dense(num_classes, activation='softmax')
    ])

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

# GRU Model 
def build_gru_model(input_shape, num_classes):
    model = Sequential([
        GRU(GRU_UNITS_1, return_sequences=True, input_shape=input_shape),
        BatchNormalization(),
        Dropout(DROPOUT_RATE),

        GRU(GRU_UNITS_2, return_sequences=False),
        BatchNormalization(),
        Dropout(DROPOUT_RATE),

        Dense(DENSE_UNITS, activation='relu'),
        Dropout(DENSE_DROPOUT),

        Dense(num_classes, activation='softmax')
    ])

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

lstm_model = build_lstm_model(input_shape, num_classes)
gru_model = build_gru_model(input_shape, num_classes)

print("\nLSTM Model Summary:")
lstm_model.summary()

print("\nGRU Model Summary:")
gru_model.summary()

In [ ]:
# TRAINING

# Callbacks 
lstm_callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=PATIENCE,
        restore_best_weights=True,
        verbose=1
    ),
    ModelCheckpoint(
        filepath='lstm_best_model.keras',
        monitor='val_loss',
        save_best_only=True,
        verbose=0
    )
]

gru_callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=PATIENCE,
        restore_best_weights=True,
        verbose=1
    ),
    ModelCheckpoint(
        filepath='gru_best_model.keras',
        monitor='val_loss',
        save_best_only=True,
        verbose=0
    )
]

# Train LSTM 
print("="*50)
print("Training LSTM model...")
print("="*50)

lstm_history = lstm_model.fit(
    X_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=VALIDATION_SPLIT,
    callbacks=lstm_callbacks,
    verbose=1
)

lstm_stopped_epoch = len(lstm_history.history['loss'])
lstm_best_val_acc = max(lstm_history.history['val_accuracy'])
lstm_best_val_loss = min(lstm_history.history['val_loss'])

print(f"\nLSTM Training complete!")
print(f"  Stopped at epoch:        {lstm_stopped_epoch}")
print(f"  Best validation accuracy: {lstm_best_val_acc:.4f} ({lstm_best_val_acc*100:.1f}%)")
print(f"  Best validation loss:     {lstm_best_val_loss:.4f}")

# Train GRU 
print("\n" + "="*50)
print("Training GRU model...")
print("="*50)

gru_history = gru_model.fit(
    X_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=VALIDATION_SPLIT,
    callbacks=gru_callbacks,
    verbose=1
)

gru_stopped_epoch = len(gru_history.history['loss'])
gru_best_val_acc = max(gru_history.history['val_accuracy'])
gru_best_val_loss = min(gru_history.history['val_loss'])

print(f"\nGRU Training complete!")
print(f"  Stopped at epoch:         {gru_stopped_epoch}")
print(f"  Best validation accuracy: {gru_best_val_acc:.4f} ({gru_best_val_acc*100:.1f}%)")
print(f"  Best validation loss:     {gru_best_val_loss:.4f}")

# Quick training comparison 
print("\n" + "="*50)
print("Training Summary")
print("="*50)
print(f"{'':20} {'LSTM':>10} {'GRU':>10}")
print("-"*40)
print(f"{'Stopped at epoch':<20} {lstm_stopped_epoch:>10} {gru_stopped_epoch:>10}")
print(f"{'Best val accuracy':<20} {lstm_best_val_acc:>10.4f} {gru_best_val_acc:>10.4f}")
print(f"{'Best val loss':<20} {lstm_best_val_loss:>10.4f} {gru_best_val_loss:>10.4f}")

In [ ]:
# COMPARISON

# Get predictions 
print("Generating predictions...")

lstm_probs = lstm_model.predict(X_test, verbose=0)
gru_probs  = gru_model.predict(X_test, verbose=0)

lstm_preds = np.argmax(lstm_probs, axis=1)
gru_preds  = np.argmax(gru_probs,  axis=1)

# Evaluation function 
def evaluate_model(name, y_true, y_pred):
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='macro', zero_division=0)
    rec  = recall_score(y_true, y_pred, average='macro', zero_division=0)
    f1   = f1_score(y_true, y_pred, average='macro', zero_division=0)

    print(f"\n{'='*50}")
    print(f"  {name} Results")
    print(f"{'='*50}")
    print(f"  Accuracy:  {acc:.4f}  ({acc*100:.1f}%)")
    print(f"  Precision: {prec:.4f}")
    print(f"  Recall:    {rec:.4f}")
    print(f"  F1 Score:  {f1:.4f}")
    print(f"\nDetailed breakdown per patient:")
    print(classification_report(
        y_true, y_pred,
        target_names=[f'Patient {i}' for i in range(num_classes)],
        zero_division=0
    ))

    return acc, prec, rec, f1

lstm_acc, lstm_prec, lstm_rec, lstm_f1 = evaluate_model("LSTM", y_test, lstm_preds)
gru_acc,  gru_prec,  gru_rec,  gru_f1  = evaluate_model("GRU",  y_test, gru_preds)

# Side by side comparison 
print("\n" + "="*50)
print("       LSTM vs GRU — Final Comparison")
print("="*50)
print(f"{'Metric':<15} {'LSTM':>10} {'GRU':>10} {'Winner':>10}")
print("-"*50)

metrics = [
    ("Accuracy",  lstm_acc,  gru_acc),
    ("Precision", lstm_prec, gru_prec),
    ("Recall",    lstm_rec,  gru_rec),
    ("F1 Score",  lstm_f1,   gru_f1),
]

for name, lstm_val, gru_val in metrics:
    winner = "LSTM" if lstm_val > gru_val else "GRU" if gru_val > lstm_val else "Tie"
    print(f"{name:<15} {lstm_val:>10.4f} {gru_val:>10.4f} {winner:>10}")

# Training curves 
print("\nPlotting training curves...")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('LSTM vs GRU — Training Curves', fontsize=16)

axes[0, 0].plot(lstm_history.history['accuracy'],     label='Train',      color='blue')
axes[0, 0].plot(lstm_history.history['val_accuracy'], label='Validation', color='orange')
axes[0, 0].set_title('LSTM — Accuracy')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Accuracy')
axes[0, 0].legend()
axes[0, 0].grid(True)

axes[0, 1].plot(lstm_history.history['loss'],     label='Train',      color='blue')
axes[0, 1].plot(lstm_history.history['val_loss'], label='Validation', color='orange')
axes[0, 1].set_title('LSTM — Loss')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Loss')
axes[0, 1].legend()
axes[0, 1].grid(True)

axes[1, 0].plot(gru_history.history['accuracy'],     label='Train', color='green')
axes[1, 0].plot(gru_history.history['val_accuracy'], label='Validation', color='red')
axes[1, 0].set_title('GRU — Accuracy')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Accuracy')
axes[1, 0].legend()
axes[1, 0].grid(True)

axes[1, 1].plot(gru_history.history['loss'],     label='Train', color='green')
axes[1, 1].plot(gru_history.history['val_loss'], label='Validation', color='red')
axes[1, 1].set_title('GRU — Loss')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Loss')
axes[1, 1].legend()
axes[1, 1].grid(True)

plt.tight_layout()
plt.show()

# Confusion matrices 
print("Plotting confusion matrices...")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
patient_names = [f'Patient {i}' for i in range(num_classes)]

for ax, preds, title in zip(axes,
                             [lstm_preds, gru_preds],
                             ['LSTM Confusion Matrix', 'GRU Confusion Matrix']):
    cm = confusion_matrix(y_test, preds)
    im = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    ax.set_title(title)
    ax.set_xlabel('Predicted Patient')
    ax.set_ylabel('Actual Patient')
    ax.set_xticks(range(num_classes))
    ax.set_yticks(range(num_classes))
    ax.set_xticklabels(patient_names, rotation=45, ha='right')
    ax.set_yticklabels(patient_names)

    for i in range(num_classes):
        for j in range(num_classes):
            ax.text(j, i, str(cm[i, j]),
                    ha='center', va='center',
                    color='white' if cm[i, j] > cm.max()/2 else 'black',
                    fontsize=8)

    plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.show()

# Save everything for later use 
print("\nSaving models, test data and histories...")

lstm_model.save('lstm_best_model.keras')
gru_model.save('gru_best_model.keras')
np.save('X_test.npy', X_test)
np.save('y_test.npy', y_test)
np.save('lstm_history.npy', lstm_history.history)
np.save('gru_history.npy',  gru_history.history)

print("Saved:")
print("  lstm_best_model.keras")
print("  gru_best_model.keras")
print("  X_test.npy")
print("  y_test.npy")
print("  lstm_history.npy")
print("  gru_history.npy")

In [ ]:
# TESTING ON UNSEEN RECORDINGS

UNSEEN_PER_PATIENT = 3

print(f"Finding {UNSEEN_PER_PATIENT} unseen recordings per patient...")

# Find recordings not used in training 
unseen_rows = []

for patient_id in top_patients:
    patient_df = df[df['subject_id'] == patient_id]
    unseen = patient_df[~patient_df['path'].isin(already_loaded_paths)]
    
    if len(unseen) == 0:
        print(f"  WARNING — No unseen recordings available for Patient {patient_id}")
        continue
    
    available = min(UNSEEN_PER_PATIENT, len(unseen))
    if available < UNSEEN_PER_PATIENT:
        print(f"  WARNING — Patient {patient_id}: only {available} unseen recordings "
              f"available instead of {UNSEEN_PER_PATIENT}")
    
    sampled = unseen.sample(n=available, random_state=None)
    for _, row in sampled.iterrows():
        unseen_rows.append(row)
    
    print(f"  Patient {patient_id}: {len(unseen)} unseen available, taking {available}")

unseen_df = pd.DataFrame(unseen_rows).reset_index(drop=True)
print(f"\nTotal unseen recordings to test: {len(unseen_df)}")

# Load the unseen recordings 
print("\nLoading unseen recordings...")

unseen_signals = []
unseen_labels = []
unseen_recording_ids = []

for _, row in unseen_df.iterrows():
    record_path = f"{BASE_PATH}/{row['path'].replace('files/', '')}"
    try:
        record = wfdb.rdrecord(record_path)
        unseen_signals.append(record.p_signal.copy())
        unseen_labels.append(row['subject_id'])
        unseen_recording_ids.append(row['subject_id'])
        print(f"  Loaded unseen recording for Patient {row['subject_id']}")
    except Exception as e:
        print(f"  Failed to load {record_path}: {e}")

print(f"\nSuccessfully loaded {len(unseen_signals)} unseen recordings")

# Check and fix NaN values 
print("\nChecking unseen recordings for NaN values...")
for i, signal in enumerate(unseen_signals):
    if np.isnan(signal).any():
        for lead in range(signal.shape[1]):
            lead_mean = np.nanmean(signal[:, lead])
            nan_mask = np.isnan(signal[:, lead])
            signal[nan_mask, lead] = lead_mean
        unseen_signals[i] = signal
        print(f"  Fixed NaN values in unseen recording {i}")
print("NaN check complete!")

# Normalize 
print("\nNormalizing unseen recordings...")
unseen_normalized = []

for signal in unseen_signals:
    norm_signal = np.zeros_like(signal)
    for lead in range(signal.shape[1]):
        lead_signal = signal[:, lead]
        mean = np.mean(lead_signal)
        std = np.std(lead_signal)
        if std == 0:
            norm_signal[:, lead] = 0
        else:
            norm_signal[:, lead] = (lead_signal - mean) / std
    unseen_normalized.append(norm_signal)

# Segment into windows 
print("Segmenting into windows...")
unseen_windows = []
unseen_window_labels = []
unseen_window_recording_idx = []

for i, signal in enumerate(unseen_normalized):
    start = 0
    while start + WINDOW_SIZE <= signal.shape[0]:
        unseen_windows.append(signal[start:start + WINDOW_SIZE])
        unseen_window_labels.append(unseen_labels[i])
        unseen_window_recording_idx.append(i)
        start += STEP_SIZE

unseen_windows = np.array(unseen_windows)
print(f"Total unseen windows: {len(unseen_windows)}")

# Encode labels 
unseen_encoded = le.transform(unseen_window_labels)

# Get predictions 
print("\nGenerating predictions...")
lstm_unseen_probs = lstm_model.predict(unseen_windows, verbose=0)
gru_unseen_probs  = gru_model.predict(unseen_windows, verbose=0)

lstm_unseen_preds = np.argmax(lstm_unseen_probs, axis=1)
gru_unseen_preds  = np.argmax(gru_unseen_probs,  axis=1)

# Majority vote per recording 
from scipy import stats

lstm_recording_preds = []
gru_recording_preds  = []
actual_recording_labels = []

for i in range(len(unseen_signals)):
    window_mask = np.array(unseen_window_recording_idx) == i

    lstm_windows_preds = lstm_unseen_preds[window_mask]
    gru_windows_preds  = gru_unseen_preds[window_mask]
    actual_label       = unseen_encoded[window_mask][0]

    lstm_vote = stats.mode(lstm_windows_preds, keepdims=True).mode[0]
    gru_vote  = stats.mode(gru_windows_preds,  keepdims=True).mode[0]

    lstm_recording_preds.append(lstm_vote)
    gru_recording_preds.append(gru_vote)
    actual_recording_labels.append(actual_label)

# Print results recording by recording 
print("\n" + "="*70)
print("         UNSEEN RECORDING IDENTIFICATION RESULTS")
print("="*70)
print(f"{'Actual Patient':<20} {'LSTM Prediction':<20} {'GRU Prediction':<20} "
      f"{'LSTM':<6} {'GRU':<6}")
print("-"*70)

lstm_correct = 0
gru_correct  = 0
total = len(unseen_signals)

for i in range(total):
    actual    = actual_recording_labels[i]
    lstm_pred = lstm_recording_preds[i]
    gru_pred  = gru_recording_preds[i]

    lstm_right = "✓" if lstm_pred == actual else "✗"
    gru_right  = "✓" if gru_pred  == actual else "✗"

    if lstm_pred == actual: lstm_correct += 1
    if gru_pred  == actual: gru_correct  += 1

    actual_id    = le.inverse_transform([actual])[0]
    lstm_pred_id = le.inverse_transform([lstm_pred])[0]
    gru_pred_id  = le.inverse_transform([gru_pred])[0]

    print(f"{str(actual_id):<20} {str(lstm_pred_id):<20} "
          f"{str(gru_pred_id):<20} {lstm_right:<6} {gru_right:<6}")

# Per patient summary 
print("\n" + "="*70)
print("PER PATIENT SUMMARY")
print("="*70)
print(f"{'Patient':<15} {'LSTM':<20} {'GRU':<20}")
print("-"*70)

for patient_id in top_patients:
    encoded_id = le.transform([patient_id])[0]
    patient_mask = np.array(actual_recording_labels) == encoded_id

    if not any(patient_mask):
        continue

    patient_actual = np.array(actual_recording_labels)[patient_mask]
    patient_lstm   = np.array(lstm_recording_preds)[patient_mask]
    patient_gru    = np.array(gru_recording_preds)[patient_mask]

    lstm_score = f"{np.sum(patient_lstm == patient_actual)}/{len(patient_actual)} correct"
    gru_score  = f"{np.sum(patient_gru  == patient_actual)}/{len(patient_actual)} correct"

    print(f"{str(patient_id):<15} {lstm_score:<20} {gru_score:<20}")

# Final scores 
print("\n" + "="*70)
print("FINAL SCORES ON UNSEEN RECORDINGS")
print("="*70)
print(f"  LSTM: {lstm_correct}/{total} correct ({lstm_correct/total*100:.1f}%)")
print(f"  GRU:  {gru_correct}/{total} correct ({gru_correct/total*100:.1f}%)")

print(f"\nWindow Level Accuracy on Unseen Data:")
lstm_win_acc = accuracy_score(unseen_encoded, lstm_unseen_preds)
gru_win_acc  = accuracy_score(unseen_encoded, gru_unseen_preds)
print(f"  LSTM: {lstm_win_acc:.4f} ({lstm_win_acc*100:.1f}%)")
print(f"  GRU:  {gru_win_acc:.4f}  ({gru_win_acc*100:.1f}%)")